In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from dotenv import load_dotenv

load_dotenv('../.env')  

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
pd.set_option('display.float_format', '{:.2f}'.format)
print("Ready")

In [ ]:
from utils.db import get_connection
from models.product import Product

get_connection()
print(f"Products in DB: {Product.objects.count()}")

In [ ]:
df = Product.get_all_as_dataframe()
df.head(10)

In [ ]:
result = Product.objects(category='Electronics', quantity__gt=10)

for p in result:
    print(f"{p.sku:12} {p.name:35} qty={p.quantity}")

In [ ]:
collection = Product._get_collection()

pipeline = [
    {'$group': {
        '_id': '$category',
        'total_qty':   {'$sum': '$quantity'},
        'total_value': {'$sum': {'$multiply': ['$quantity', '$price']}},
        'count':       {'$sum': 1}
    }},
    {'$sort': {'total_qty': -1}}
]

results = list(collection.aggregate(pipeline))
df_cat  = pd.DataFrame(results).rename(columns={'_id': 'category'})
df_cat

In [ ]:
THRESHOLD = 10

plot_df  = df[['Name', 'Quantity']].copy()
plot_df  = plot_df.sort_values('Quantity')
colours  = ['#e53e3e' if q <= THRESHOLD else '#38a169'
            for q in plot_df['Quantity']]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plot_df['Name'], plot_df['Quantity'], color=colours)
ax.axvline(x=THRESHOLD, color='black', linestyle='--', label=f'Threshold ({THRESHOLD})')

red_patch   = mpatches.Patch(color='#e53e3e', label='Low Stock')
green_patch = mpatches.Patch(color='#38a169', label='OK')
ax.legend(handles=[red_patch, green_patch])

ax.set_xlabel('Quantity')
ax.set_title('Stock Levels — Red means Restock Needed', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.pivot_table(
    values='Quantity',
    index='Category',
    aggfunc=['mean', 'sum', 'count']
)
pivot.columns = ['Avg Qty', 'Total Qty', 'Products']

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Category Stats Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()